# Explore SAE features

Eyeball what a feature does: where it fires, how often, and on which positions. Needs a trained SAE
(`scripts/03_train_sae.py`) and, for the property lookup, its eval (`scripts/04_evaluate.py`).
Positions are the held-out Lichess **test** games; boards are drawn from the side to move's view
(own pieces at the bottom), which is how the net sees them.

In [ ]:
import json, os
from pathlib import Path

if not Path("outputs").exists():          # run from the repo root
    os.chdir("..")

import matplotlib.pyplot as plt
import torch

from src import hook
from src.data import lichess
from src.eval.codes import encode_corpus
from src.eval.feature_mining import density_from_codes, mine_feature
from src.sae import TopKSAE
from src.viz.features import show_top_activations

SAE_PATH = "outputs/sae/L7.pt"
meta = torch.load(SAE_PATH, map_location="cpu", weights_only=True)["meta"]
sae = TopKSAE.load(SAE_PATH)
model = hook.load_model()
fens = lichess.load_corpus("outputs/positions/lichess_2024-06.parquet", split="test", limit=2000)["fen"].tolist()
codes = encode_corpus(model, sae, meta["module"], fens)      # sparse top-k codes for every token
density = density_from_codes(codes)
print(meta["module"], "| features:", sae.config["dict_size"], "| dead on these boards:", int((density == 0).sum()))

In [ ]:
live = density[density > 0]
plt.hist(torch.log10(live).numpy(), bins=50)
plt.xlabel("log10 firing density (fraction of tokens)")
plt.ylabel("features")
plt.title("How often each feature fires");

## Features the eval matched to a chess property

In [ ]:
results = json.loads(Path("outputs/results", Path(SAE_PATH).stem, "results.json").read_text())
for r in sorted(results["properties"], key=lambda r: -r["test_f1"])[:25]:
    print(f"{r['property']:26s} feature {r['feature']:5d}  held-out F1 {r['test_f1']:.2f}")

In [ ]:
prop = "enemy_passed_pawn"                 # try: own_king_in_check, enemy_hanging, own_knight ...
feature = next(r["feature"] for r in results["properties"] if r["property"] == prop)
show_top_activations(feature, mine_feature(codes, fens, feature, k=12), relative=True)

## Any feature

In [ ]:
feature = int(torch.randint(len(density), (1,)))   # or set an index by hand
print(f"feature {feature}: density {float(density[feature]):.4%}")
show_top_activations(feature, mine_feature(codes, fens, feature, k=12), relative=True)